# FEMTO / PRONOSTIA — single-modality degradation-stage baselines (vibration-only, temperature-only)

**What this notebook tests.** It sets **single-modality baseline** performance on the FEMTO/PRONOSTIA bearing dataset
under a **leave-one-operating-condition-out** protocol. It is the first experiment of a larger study. Its only job is a
trustworthy *"what does fusion need to beat"* number for **each modality on its own**.

**What it does NOT test.** There is no fusion and no combination of modalities anywhere. There is no lag-adaptive or
temporal-alignment mechanism (no HTT, no LACA), no cross-attention and no gating. Vibration and temperature are
trained and evaluated completely independently.

| Modality | Architectures | Bearings |
|---|---|---|
| Vibration (2 accelerometer channels, 25.6 kHz bursts) | **(A)** small encoder + 2 real `mamba_ssm` blocks + attention pooling + linear head; **(B)** WDCNN-style compact 1D-CNN | all **17** |
| Temperature (1 channel, 10 Hz) | **(A)** only (no CNN: not motivated for this signal, out of scope) | the **13** that have it |

(B) asks whether a sequence backbone earns its place on this dataset. It follows the WDCNN design: a wide first-layer
kernel, then a ~50-sample second kernel, global pooling and one FC layer (Zhang et al., 2017, *"A New Deep Learning
Model for Fault Diagnosis with Good Anti-Noise and Domain Adaptation Ability on Raw Vibration Signals"*, Sensors
17(2)). WDCNN is the field-standard raw-vibration baseline.

**Self-sufficient, single Run All.** The notebook installs its own dependencies (`mamba_ssm`), downloads the archive,
extracts it recursively and parses the raw CSVs. It writes every result, prediction file and checkpoint to
`/kaggle/working`. It reads nothing from any other notebook or attached Kaggle dataset. It needs a GPU with internet
on.

## Dataset, citation and license

**FEMTO Bearing Data Set, NASA Prognostics Data Repository, NASA Ames Research Center.**
P. Nectoux, R. Gouriveau, K. Medjaher, E. Ramasso, B. Chebel-Morello, N. Zerhouni, C. Varnier, *"PRONOSTIA: An
experimental platform for bearings accelerated degradation tests"*, IEEE International Conference on Prognostics and
Health Management (PHM 2012 Prognostic Challenge), 2012.

The archive has **no license file**. The NASA Prognostics Data Repository it comes from operates under
**CC BY 3.0 US**: unrestricted use, distribution and reproduction, with citation required (above).

Download source: the PHM Society mirror `https://phm-datasets.s3.amazonaws.com/NASA/10.+FEMTO+Bearing.zip`
(about 1157 MB). NASA's own site has intermittent direct-download problems, so it is not used.

## Methodological choices (stated before any result)

**1. The label is our own proxy, not something the dataset defines.** FEMTO is run-to-failure. It ships **no
ground-truth fault-onset or degradation-stage label of any kind**: no readme, label file or metadata in the archive.
Only the *end* of each recorded sequence means anything (the bearing was stopped, generally when vibration amplitude
passed a safety threshold). **As a stated methodological choice**, each window gets a **degradation-stage proxy
label**. The label is the window's position in **its own bearing's** recorded lifetime, split into `N_STAGES` (default
4) **equal-width bins of elapsed time**: Q1 is the first 25 % of that bearing's recorded life, and Q4 is the last 25 %,
ending at the bearing's last recorded record. Elapsed time is measured from each file's own timestamp.
Stage boundaries are computed **per bearing**, never on a pool of bearings. `N_STAGES` is one named constant and
nothing else depends on its value. Each modality's label uses that modality's own record timeline: vibration from its
first to its last acc burst, temperature from its first to its last temp record.

**2. Only `Learning_set/` and `Full_Test_Set/` are used (17 bearings).** `Test_set/` holds the *same 11 bearings* as
`Full_Test_Set/`, **truncated** before failure for the original PHM2012 blind scoring. Its Q4 would not reach
end-of-life and would corrupt the labels, so it is **excluded entirely**. The reader refuses any path under it.

**3. Leave-one-condition-out.** `Bearing1_*`, `Bearing2_*` and `Bearing3_*` are three operating conditions (about
1800 rpm / 4 kN, 1650 rpm / 4.2 kN, 1500 rpm / 5 kN). There are 3 folds. Each fold trains on 2 conditions and tests
on **all** bearings of the third. Inner validation (checkpoint selection) holds out **whole bearings**, never windows,
because windows of one trajectory are highly correlated. The rule is one bearing per training condition when that
condition has ≥ 2 bearings. Where possible it picks a bearing that has temperature, so both modalities share the
same held-out bearing. The choice is seeded and fixed per fold, and is identical across seeds and architectures.

**4. Deliberately NO cross-modal alignment.** There is a real start-time offset between vibration and temperature
inside each bearing: temperature starts up to ~107 s *after* vibration, varying per bearing (measured per bearing in the
inventory table; one bearing, 2_5, starts 0.6 s before). **This notebook never
combines the modalities, so the offset does not matter here.** The missing alignment logic is a deliberate choice,
not an oversight. The correction will matter in a later fusion notebook. The inventory table prints the offset for
information only, and nothing uses it.

**5. Four bearings have no temperature data:** `Full_Test_Set/Bearing1_3`, `Bearing2_3`, `Learning_set/Bearing2_2`
and `Bearing3_2` (zero `temp_*.csv` files). The pre-notebook audit also listed `Full_Test_Set/Bearing2_6`, but the
notebook's own file sweep finds **116 temp files** there: semicolon-delimited, 600 rows each, acc:temp ratio 6.04 and a
start offset in line with the other bearings. So it is real temperature data and is used. The notebook still stops
loudly if the observed list ever differs from `EXPECTED_NO_TEMP`. Vibration uses all **17** bearings and temperature
uses **13**. Both counts are printed next to every result. Vibration's bearing set is **never** cut down to match
temperature's.

**6. A window is one acquisition record.** For vibration, one `acc_*.csv` burst: 2560 samples × 2 channels = 0.1 s at
25.6 kHz, one burst every 10 s. For temperature, one `temp_*.csv` record (its row count is measured below). So **no
window can cross a bearing boundary**. Normalization is a per-channel z-score whose mean and SD come from **training
bearings only**, separately for each fold. Vibration downsampling is a named flag, `VIB_DOWNSAMPLE_FACTOR = 1`, and is
OFF by default.

In [ ]:
# ==============================================================================
# 1. SETUP + CONSTANTS — every fixed value lives here
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS; must be set before CUDA initializes
import sys, re, json, time, math, copy, random, shutil, zipfile, hashlib, subprocess, urllib.request
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_T0 = time.time()

# --- paths: results/probs/checkpoints -> /kaggle/working (persisted); download/extraction/parse cache -> scratch ---
ON_KAGGLE = Path("/kaggle/working").is_dir()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "femto_working"
SCRATCH_DIR = Path("/tmp/femto") if ON_KAGGLE else WORK_DIR / "_scratch"
DOWNLOAD_PATH = SCRATCH_DIR / "download" / "femto_bearing.zip"
EXTRACT_DIR = SCRATCH_DIR / "extracted"
CACHE_DIR = SCRATCH_DIR / "parsed"
RESULTS_CSV = WORK_DIR / "femto_baseline_results.csv"
SUMMARY_CSV = WORK_DIR / "femto_baseline_summary.csv"
PROBS_DIR = WORK_DIR / "femto_probs"
CKPT_DIR = WORK_DIR / "femto_checkpoints"
for _d in (WORK_DIR, SCRATCH_DIR, CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- download / extraction ---
FEMTO_URL = "https://phm-datasets.s3.amazonaws.com/NASA/10.+FEMTO+Bearing.zip"   # PHM Society mirror (NOT nasa.gov)
EXPECTED_ZIP_MB = 1157
MAX_EXTRACTION_PASSES = 10

# --- splits / bearings (quirks 3, 4, 7) ---
USE_SPLITS = ["Learning_set", "Full_Test_Set"]
EXCLUDED_SPLIT = "Test_set"                   # truncated copy of Full_Test_Set; never read
EXPECTED_SPLIT_BEARINGS = {
    "Learning_set": {"Bearing1_1", "Bearing1_2", "Bearing2_1", "Bearing2_2", "Bearing3_1", "Bearing3_2"},
    "Full_Test_Set": {"Bearing1_3", "Bearing1_4", "Bearing1_5", "Bearing1_6", "Bearing1_7",
                      "Bearing2_3", "Bearing2_4", "Bearing2_5", "Bearing2_6", "Bearing2_7", "Bearing3_3"},
}
EXPECTED_BEARINGS = set().union(*EXPECTED_SPLIT_BEARINGS.values())
# Audit listed Bearing2_6 too, but this notebook's sweep finds 116 real temp files there (ratio 6.04, 600 rows/file).
EXPECTED_NO_TEMP = {"Bearing1_3", "Bearing2_3", "Bearing2_2", "Bearing3_2"}
ALLOW_NO_TEMP_MISMATCH = False                # True = continue (loudly) if the zero-temperature list differs
CONDITIONS = {1: "~1800 rpm / 4.0 kN", 2: "~1650 rpm / 4.2 kN", 3: "~1500 rpm / 5.0 kN"}

# --- file formats (quirks 5, 6) ---
ACC_NCOLS, ACC_ROWS, ACC_FS = 6, 2560, 25_600     # hour, minute, second, microsecond, horiz_g, vert_g
ACC_SIGNAL_COLS = [4, 5]
ACC_DELIMS = (",", ";")                          # try comma first, then semicolon (auto-detected PER FILE)
TEMP_NCOLS, TEMP_FS = 5, 10                      # hour; minute; second; subsecond_index (0-9); temperature_C
TEMP_SIGNAL_COL = 4
TEMP_DELIMS = (";", ",")                         # documented semicolon; auto-detected per file all the same
ACC_PERIOD_S, TEMP_PERIOD_S = 10, 60             # documented acquisition periods -> expected acc:temp ≈ 6:1
RATIO_TOL = (5.0, 7.0)

# --- task / windows ---
N_STAGES = 4                                     # degradation-stage proxy bins (equal-width in elapsed time, per bearing)
VIB_DOWNSAMPLE_FACTOR = 1                        # OFF by default; >1 applies scipy.signal.decimate (anti-aliased)

# --- run matrix ---
SEEDS = [42, 123]
INNER_VAL_SEED = 2026                            # bearing-level inner-validation choice (fixed per fold)
FOLD_CONDITIONS = [1, 2, 3]                      # the held-out (test) condition of each fold
RUNS = [("vibration", "mamba"), ("vibration", "cnn"), ("temperature", "mamba")]

# --- (A) encoder + Mamba (prior UnimodalMamba pattern) ---
D_MODEL = 256
ENC_HIDDEN = 64
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
POOL_HIDDEN = 64
DROPOUT = 0.15

# --- (B) WDCNN-style 1D-CNN (vibration only) ---
CNN_K1, CNN_S1, CNN_C1 = 64, 16, 16              # wide first kernel (WDCNN: 64, stride 16, 16 filters)
CNN_POOL1 = 2
CNN_K2, CNN_C2 = 50, 32                          # ~50-sample second kernel

# --- optimization (prior work's recipe; CNN uses WDCNN's customary 1e-3 learning rate) ---
EPOCHS = 20
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = {"mamba": 1e-4, "cnn": 1e-3}
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 5                          # stop after this many epochs without a new best val macro-F1
NUM_WORKERS = 2
NORM_EPS = 1e-6

# A results row / prediction file counts as "done" only under the SAME config tag (changing N_STAGES etc. re-runs).
CONFIG_TAG = f"S{N_STAGES}_ds{VIB_DOWNSAMPLE_FACTOR}_e{EPOCHS}_p{EARLY_STOP_PATIENCE}_v{INNER_VAL_SEED}"

print(f"Kaggle: {ON_KAGGLE} | outputs -> {WORK_DIR} | scratch -> {SCRATCH_DIR}")
print(f"N_STAGES {N_STAGES} | VIB_DOWNSAMPLE_FACTOR {VIB_DOWNSAMPLE_FACTOR} | seeds {SEEDS} | runs {RUNS} | "
      f"config tag {CONFIG_TAG}")

In [ ]:
# ==============================================================================
# 2. REAL mamba-ssm — install if missing, import, raise on failure (no fallback)
# ==============================================================================
import importlib, importlib.util
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU: real mamba-ssm needs a GPU. Enable a GPU accelerator and re-run.")
if importlib.util.find_spec("mamba_ssm") is None:
    _ok = (_pip(["packaging", "ninja"]) and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output). Check internet + GPU, restart, re-run.")
try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"`from mamba_ssm import Mamba` failed: {exc!r}. No fallback is used.") from exc
assert MAMBA_CLASS.__module__.startswith("mamba_ssm"), f"Mamba class comes from {MAMBA_CLASS.__module__}, not mamba_ssm"
device = torch.device("cuda")
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} ({MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__}) | torch "
      f"{torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 3. SEEDING, DETERMINISM FLAGS, HOOK HELPER
# ==============================================================================
def seed_all(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
seed_all(42)

HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=True):
    ok = bool(ok)
    HOOKS[name] = [(ok, msg, stop)]                  # re-running a cell replaces that hook's earlier result
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)

## Data acquisition: download → recursive extraction → split discovery

The outer zip holds `10. FEMTO Bearing/FEMTOBearingDataSet.zip`. That zip holds `Training_set.zip`, `Test_set.zip`
and `Validation_Set.zip`. Extraction is **recursive**: each pass finds every `.zip` left in the tree, extracts it in
place and removes it. Passes repeat until no `.zip` remains. The downloaded archive is kept outside the extraction
tree, so a rerun in the same session neither downloads nor extracts again.

In [ ]:
# ==============================================================================
# 4. DOWNLOAD (PHM Society mirror of the NASA Prognostics Data Repository)
# ==============================================================================
def download(url, dest, retries=3):
    if dest.exists() and zipfile.is_zipfile(dest):
        print(f"Archive already present (re-used): {dest} ({dest.stat().st_size / 1e6:,.0f} MB)")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    for attempt in range(1, retries + 1):
        try:
            print(f"Downloading {url} (attempt {attempt}/{retries}) ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, open(part, "wb") as f:
                total = int(r.headers.get("Content-Length") or 0)
                done, next_report = 0, 0
                while True:
                    chunk = r.read(8 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
                    done += len(chunk)
                    if done >= next_report:
                        print(f"  {done / 1e6:8,.0f} / {total / 1e6:,.0f} MB")
                        next_report += 100_000_000
            if total and done != total:
                raise IOError(f"truncated download: {done:,} of {total:,} bytes")
            part.replace(dest)
            return
        except Exception as exc:
            print(f"  attempt {attempt} failed: {exc!r}")
            if attempt == retries:
                raise RuntimeError(f"Download of {url} failed after {retries} attempts (is Kaggle internet ON?)") from exc
            time.sleep(15 * attempt)


_t = time.time()
download(FEMTO_URL, DOWNLOAD_PATH)
_size_mb, _size_mib = DOWNLOAD_PATH.stat().st_size / 1e6, DOWNLOAD_PATH.stat().st_size / 2 ** 20
hook("download_is_zip", zipfile.is_zipfile(DOWNLOAD_PATH), f"{DOWNLOAD_PATH.name} is a valid zip ({time.time() - _t:.0f} s)")
hook("download_size", min(abs(_size_mb - EXPECTED_ZIP_MB), abs(_size_mib - EXPECTED_ZIP_MB)) / EXPECTED_ZIP_MB < 0.10,
     f"{_size_mb:,.0f} MB ({_size_mib:,.0f} MiB) vs documented ~{EXPECTED_ZIP_MB} MB", stop=False)

In [ ]:
# ==============================================================================
# 5. RECURSIVE EXTRACTION — extract every .zip found, in passes, until none remain
# ==============================================================================
def is_junk(p):
    """macOS resource-fork artefacts (__MACOSX/, ._name) are not data and not real archives."""
    p = Path(p)
    return "__MACOSX" in p.parts or p.name.startswith("._")


def zips_in(root):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".zip")


SENTINEL = EXTRACT_DIR / ".femto_extraction_complete.json"
if SENTINEL.exists():
    EXTRACTION_LOG = json.loads(SENTINEL.read_text())
    print(f"Extraction already complete (sentinel present) — re-using {EXTRACT_DIR}")
else:
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)                       # a half-finished earlier extraction is redone from scratch
    EXTRACT_DIR.mkdir(parents=True)
    _t = time.time()
    with zipfile.ZipFile(DOWNLOAD_PATH) as z:
        z.extractall(EXTRACT_DIR)
        EXTRACTION_LOG = [{"pass": 1, "zip": DOWNLOAD_PATH.name, "members": len(z.namelist())}]
    n_pass = 1
    while True:
        found = zips_in(EXTRACT_DIR)
        for p in [p for p in found if is_junk(p)]:
            p.unlink()
        found = [p for p in found if not is_junk(p)]
        if not found:
            break
        n_pass += 1
        if n_pass > MAX_EXTRACTION_PASSES:
            raise RuntimeError(f"Still finding .zip files after {MAX_EXTRACTION_PASSES} passes: {found[:5]}")
        for zp in found:
            with zipfile.ZipFile(zp) as z:
                z.extractall(zp.parent)
                n_members = len(z.namelist())
            EXTRACTION_LOG.append({"pass": n_pass, "zip": str(zp.relative_to(EXTRACT_DIR)), "members": n_members})
            zp.unlink()                                  # expanded in place; removing it is what lets the loop end
    SENTINEL.write_text(json.dumps(EXTRACTION_LOG))
    print(f"Extraction finished in {time.time() - _t:.0f} s")

for e in EXTRACTION_LOG:
    print(f"  pass {e['pass']}: {e['zip']}  ({e['members']:,} members)")
_remaining = [p for p in zips_in(EXTRACT_DIR) if not is_junk(p)]
_n_passes = max(e["pass"] for e in EXTRACTION_LOG)
hook("recursive_extraction", not _remaining,
     f"{len(EXTRACTION_LOG)} archives extracted over {_n_passes} passes; .zip files remaining anywhere: {len(_remaining)}")
hook("nested_depth", _n_passes >= 3, f"{_n_passes} extraction passes (documented nesting needs >= 3)", stop=False)

In [ ]:
# ==============================================================================
# 6. SPLIT DISCOVERY — Learning_set + Full_Test_Set only; Test_set located and EXCLUDED
# ==============================================================================
BEARING_RE = re.compile(r"^Bearing(\d)_(\d+)$")


def bearing_children(d):
    return sorted(c for c in d.iterdir() if c.is_dir() and BEARING_RE.match(c.name))


ALL_DIRS = [Path(r) for r, _, _ in os.walk(EXTRACT_DIR) if not is_junk(r)]


def find_split(name):
    """Directories named `name` (case-insensitive) that directly contain Bearing*_* folders."""
    return [d for d in ALL_DIRS if d.name.lower() == name.lower() and bearing_children(d)]


SPLIT_ROOTS = {}
for _name in USE_SPLITS:
    _found = find_split(_name)
    hook(f"split_{_name}", len(_found) == 1,
         f"{_name}/ found exactly once: {[str(p.relative_to(EXTRACT_DIR)) for p in _found]}")
    SPLIT_ROOTS[_name] = _found[0]

# Test_set: only its folder NAMES are listed (to show it mirrors Full_Test_Set). No file under it is ever opened.
_excluded = find_split(EXCLUDED_SPLIT)
for _d in _excluded:
    print(f"EXCLUDED {_d.relative_to(EXTRACT_DIR)}/ (truncated PHM2012 copy) — bearing folders: "
          f"{[c.name for c in bearing_children(_d)]} — none of its files are read")
if not _excluded:
    print(f"Note: no {EXCLUDED_SPLIT}/ folder found (nothing to exclude).")

BEARINGS, _dups = {}, []
for _split, _root in SPLIT_ROOTS.items():
    for _d in bearing_children(_root):
        if _d.name in BEARINGS:
            _dups.append(_d.name)
        _m = BEARING_RE.match(_d.name)
        BEARINGS[_d.name] = {"split": _split, "dir": _d, "condition": int(_m.group(1)), "index": int(_m.group(2))}
BEARING_NAMES = sorted(BEARINGS, key=lambda b: (BEARINGS[b]["condition"], BEARINGS[b]["index"]))
COND_OF = {b: BEARINGS[b]["condition"] for b in BEARING_NAMES}

hook("17_bearings", len(BEARINGS) == 17 and set(BEARINGS) == EXPECTED_BEARINGS and not _dups,
     f"{len(BEARINGS)} distinct bearings across {USE_SPLITS} (duplicates across splits: {_dups}); "
     f"missing {sorted(EXPECTED_BEARINGS - set(BEARINGS))}, unexpected {sorted(set(BEARINGS) - EXPECTED_BEARINGS)}")
for _split, _exp in EXPECTED_SPLIT_BEARINGS.items():
    _got = {b for b in BEARINGS if BEARINGS[b]["split"] == _split}
    hook(f"split_members_{_split}", _got == _exp, f"{_split}: {len(_got)} bearings {sorted(_got)}")
hook("excluded_split_not_used", all(EXCLUDED_SPLIT.lower() not in [p.lower() for p in BEARINGS[b]["dir"].parts]
                                    for b in BEARINGS), f"no bearing directory lies under {EXCLUDED_SPLIT}/")
for _c, _desc in CONDITIONS.items():
    print(f"  condition {_c} ({_desc}): {[b for b in BEARING_NAMES if COND_OF[b] == _c]}")

## Parsing: per-file delimiter auto-detection, shape/NaN checks, timestamps

Every CSV is parsed with **per-file delimiter auto-detection**. Accelerometer files try comma first; if that does not
give 6 columns, they retry with semicolon. Temperature files try semicolon (documented) first, then comma, and need 5
columns. Nothing is special-cased by folder name. `Full_Test_Set/Bearing1_4`'s semicolon-delimited acc files are
expected to be caught by this rule, and any other such file is reported. Every acc file must have exactly 2560
finite rows. Every temp file is checked for NaN, and its row count is **measured** and printed rather than assumed.
A file's time is its first row's `hour*3600 + minute*60 + second + subsecond`. A midnight rollover, if any, is
unwrapped. **Clock glitches are repaired, never ignored.** When a file's time is ≤ the previous file's time, the
repair looks ahead (up to `MAX_EXCURSION_FILES` files) for the first file whose time is again later than the last
good one, i.e. back on the original timeline. The files in between have **corrupted timestamps**, and their times are
linearly interpolated between the two good neighbours. Every other timestamp, including genuine forward gaps, is left
as recorded. Example: Bearing1_1 acc files 2121–2122 both read 09:38:46 inside an otherwise continuous 15:32–15:33
run. If the clock never comes back (a persistent reset), that single step is set to the bearing's median step and the
rest of the timeline is shifted. Each repair is printed with file numbers, raw timestamps and the times it was given.
The stage labels of a repaired bearing are compared with file-order labels. More than `MAX_CLOCK_REPAIRS_PER_BEARING`
repair events on one bearing means it is not an isolated glitch, and the notebook stops. After repair, each bearing's
timeline must be strictly increasing. A **span check** (`recorded span ≈ (n_files − 1) × median period`) flags any
bearing whose timeline length disagrees with its file count. Parsed arrays are cached in scratch space so a
rerun in the same session skips re-parsing.

In [ ]:
# ==============================================================================
# 7. RAW CSV PARSING (guarded against Test_set, delimiter auto-detected per file, cached)
# ==============================================================================
PARSER_VERSION = 1
READ_COUNT = Counter()                       # files read, by split


def guard_not_excluded(path):
    if EXCLUDED_SPLIT.lower() in [p.lower() for p in Path(path).parts]:
        raise RuntimeError(f"Refusing to read {path}: it is under the excluded {EXCLUDED_SPLIT}/ split")


def read_csv_autodelim(path, expected_ncols, delimiters, split):
    """Try each delimiter in order; return the first parse with exactly `expected_ncols` columns."""
    guard_not_excluded(path)
    for sep in delimiters:
        df = pd.read_csv(path, header=None, sep=sep, engine="c")
        if df.shape[1] == expected_ncols:
            READ_COUNT[split] += 1
            return df.to_numpy(dtype=np.float64), sep
    raise ValueError(f"{path}: none of the delimiters {delimiters} gives {expected_ncols} columns")


def file_num(p):
    return int(re.search(r"_(\d+)\.csv$", p.name).group(1))


def parse_bearing(name):
    info = BEARINGS[name]
    cache = CACHE_DIR / f"{name}_p{PARSER_VERSION}.npz"
    if cache.exists():
        with np.load(cache, allow_pickle=False) as z:
            out = {k: z[k] for k in z.files}
        READ_COUNT[info["split"] + " (cached)"] += len(out["acc_t_raw"]) + len(out["temp_t_raw"])
        return out
    acc_files = sorted(info["dir"].glob("acc_*.csv"), key=file_num)
    temp_files = sorted(info["dir"].glob("temp_*.csv"), key=file_num)
    acc_x = np.full((len(acc_files), ACC_ROWS, len(ACC_SIGNAL_COLS)), np.nan, np.float32)
    acc_t, acc_sep, acc_bad = np.full(len(acc_files), np.nan), [], []
    for i, f in enumerate(acc_files):
        a, sep = read_csv_autodelim(f, ACC_NCOLS, ACC_DELIMS, info["split"])
        acc_sep.append(sep)
        n_bad = int((~np.isfinite(a)).sum())
        if a.shape[0] != ACC_ROWS or n_bad:
            acc_bad.append(f"{f.name}: rows {a.shape[0]} (expected {ACC_ROWS}), non-finite values {n_bad}")
            continue
        acc_x[i] = a[:, ACC_SIGNAL_COLS]
        acc_t[i] = a[0, 0] * 3600 + a[0, 1] * 60 + a[0, 2] + a[0, 3] * 1e-6
    temp_chunks, temp_len, temp_t, temp_sep, temp_bad = [], [], np.full(len(temp_files), np.nan), [], []
    for i, f in enumerate(temp_files):
        a, sep = read_csv_autodelim(f, TEMP_NCOLS, TEMP_DELIMS, info["split"])
        temp_sep.append(sep)
        n_bad = int((~np.isfinite(a)).sum())
        if n_bad or a.shape[0] == 0:
            temp_bad.append(f"{f.name}: rows {a.shape[0]}, non-finite values {n_bad}")
        temp_chunks.append(a[:, TEMP_SIGNAL_COL].astype(np.float32))
        temp_len.append(a.shape[0])
        if a.shape[0]:
            temp_t[i] = a[0, 0] * 3600 + a[0, 1] * 60 + a[0, 2] + a[0, 3] / TEMP_FS
    out = {
        "acc_x": acc_x, "acc_t_raw": acc_t, "acc_num": np.array([file_num(f) for f in acc_files], np.int64),
        "acc_sep": np.array(acc_sep, dtype="<U1"), "acc_bad": np.array(acc_bad, dtype=str),
        "temp_flat": np.concatenate(temp_chunks) if temp_chunks else np.zeros(0, np.float32),
        "temp_len": np.array(temp_len, np.int64), "temp_t_raw": temp_t,
        "temp_num": np.array([file_num(f) for f in temp_files], np.int64),
        "temp_sep": np.array(temp_sep, dtype="<U1"), "temp_bad": np.array(temp_bad, dtype=str),
    }
    np.savez(cache, **out)
    return out


DATA = {}
_t = time.time()
for _b in BEARING_NAMES:
    _tb = time.time()
    DATA[_b] = parse_bearing(_b)
    print(f"  {BEARINGS[_b]['split']:>13}/{_b:<11} acc files {len(DATA[_b]['acc_t_raw']):5d} | temp files "
          f"{len(DATA[_b]['temp_t_raw']):4d} | {time.time() - _tb:5.1f} s")
print(f"Parsed 17 bearings in {time.time() - _t:.0f} s | files read by split: {dict(READ_COUNT)}")
hook("test_set_never_read", all(EXCLUDED_SPLIT.lower() != k.split(" ")[0].lower() for k in READ_COUNT)
     and set(k.split(" ")[0] for k in READ_COUNT) <= set(USE_SPLITS),
     f"files were read only from {sorted(set(k.split(' ')[0] for k in READ_COUNT))}; {EXCLUDED_SPLIT}: 0 files")

In [ ]:
# ==============================================================================
# 8. INVENTORY + VERIFICATION: delimiters, shape/NaN, monotonic time, acc:temp ratio, zero-temperature bearings
# ==============================================================================
MAX_CLOCK_REPAIRS_PER_BEARING = 5                    # more repair events than this on one bearing -> stop
MAX_EXCURSION_FILES = 10                             # look-ahead for the clock to return to the original timeline
SPAN_TOL = 0.02                                      # |span / ((n-1) x median period) - 1| allowed by the span check


def unwrap_midnight(t):
    jumps = np.diff(t) < -43_200                     # clock went back > 12 h: a midnight rollover
    return t + np.concatenate([[0.0], np.cumsum(jumps)]) * 86_400, int(jumps.sum())


def hms(sec):
    return f"{int(sec // 3600):02d}:{int(sec % 3600 // 60):02d}:{sec % 60:09.6f}"


def repair_clock(t_raw, nums, label):
    """(1) Unwrap midnight rollovers. (2) At each non-positive step after file i, find the first later file j (within
    MAX_EXCURSION_FILES) whose time is > t[i]: files i+1..j-1 carry corrupted timestamps and are linearly
    interpolated between t[i] and t[j]. If the clock never returns, treat it as a persistent reset: set that one step
    to the median step and shift the rest of the timeline. All other timestamps are left as recorded."""
    if len(t_raw) < 2 or not np.all(np.isfinite(t_raw)):
        return t_raw.copy(), 0, []                   # non-finite times are left for the shape_nan / monotonic hooks
    t, wraps = unwrap_midnight(t_raw)
    t = t.copy()
    d = np.diff(t)
    med = float(np.median(d[d > 0]))
    repairs, i = [], 0
    while i < len(t) - 1:
        if t[i + 1] > t[i]:
            i += 1
            continue
        ahead = np.where(t[i + 1:i + 2 + MAX_EXCURSION_FILES] > t[i])[0]
        if len(ahead):
            j = i + 1 + int(ahead[0])
            new = np.interp(np.arange(i + 1, j), [i, j], [t[i], t[j]])
            repairs.append(
                f"{label}: CORRUPTED TIMESTAMPS on {j - i - 1} file(s) {nums[i + 1:j].tolist()} between good files "
                f"{nums[i]} ({hms(t_raw[i])}) and {nums[j]} ({hms(t_raw[j])}): raw "
                f"{[hms(x) for x in t_raw[i + 1:j]]} -> interpolated {[hms(x % 86_400) for x in new]} "
                f"(step {(t[j] - t[i]) / (j - i):.3f} s; bearing median {med:.3f} s)")
            t[i + 1:j] = new
            i = j
        else:
            shift = t[i] + med - t[i + 1]
            repairs.append(
                f"{label}: PERSISTENT CLOCK RESET after file {nums[i]}: raw {hms(t_raw[i])} -> {hms(t_raw[i + 1])} "
                f"(step {t[i + 1] - t[i]:+.3f} s, no return within {MAX_EXCURSION_FILES} files) -> step set to median "
                f"{med:.3f} s, the remaining {len(t) - i - 1} files shifted by {shift:+.3f} s")
            t[i + 1:] += shift
            i += 1
    return t, wraps, repairs


_rows, _bad_files, _nonmono = [], [], []
CLOCK_REPAIRS = {}
for _b in BEARING_NAMES:
    D = DATA[_b]
    D["acc_t"], _aw, _ar = repair_clock(D["acc_t_raw"], D["acc_num"], f"{_b} acc")
    D["temp_t"], _tw, _tr = repair_clock(D["temp_t_raw"], D["temp_num"], f"{_b} temp")
    if _ar or _tr:
        CLOCK_REPAIRS[_b] = {"acc": _ar, "temp": _tr}
    for _arr, _kind in ((D["acc_t"], "acc"), (D["temp_t"], "temp")):
        if len(_arr) > 1 and not (np.all(np.isfinite(_arr)) and np.all(np.diff(_arr) > 0)):
            _nonmono.append(f"{_b} {_kind}")
    _bad_files += [f"{_b}/{s}" for s in list(D["acc_bad"]) + list(D["temp_bad"])]
    _n_acc, _n_temp = len(D["acc_t"]), len(D["temp_t"])
    _rows.append({
        "bearing": _b, "split": BEARINGS[_b]["split"], "cond": COND_OF[_b],
        "n_acc": _n_acc, "n_temp": _n_temp,
        "acc_comma": int((D["acc_sep"] == ",").sum()), "acc_semicolon": int((D["acc_sep"] == ";").sum()),
        "temp_semicolon": int((D["temp_sep"] == ";").sum()), "temp_comma": int((D["temp_sep"] == ",").sum()),
        "temp_rows": ",".join(str(v) for v in sorted(set(D["temp_len"].tolist()))) or "-",
        "acc:temp": round(_n_acc / _n_temp, 2) if _n_temp else np.nan,
        "acc_period_s": round(float(np.median(np.diff(D["acc_t"]))), 2) if _n_acc > 1 else np.nan,
        "temp_period_s": round(float(np.median(np.diff(D["temp_t"]))), 2) if _n_temp > 1 else np.nan,
        "life_h": round((D["acc_t"][-1] - D["acc_t"][0]) / 3600, 2),
        "midnight_wraps": _aw + _tw,
        "clock_repairs": len(_ar) + len(_tr),
        "temp_start_minus_acc_start_s (info only)": round(D["temp_t"][0] - D["acc_t"][0], 1) if _n_temp else np.nan,
    })
INVENTORY = pd.DataFrame(_rows)
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(INVENTORY.to_string(index=False))
print(f"Totals: {INVENTORY.n_acc.sum():,} acc files, {INVENTORY.n_temp.sum():,} temp files")

# --- shape / NaN safety net, monotonic timestamps ---
hook("shape_nan", not _bad_files, f"every acc file has {ACC_ROWS} finite rows and every temp file is finite; "
     f"violations: {len(_bad_files)} {_bad_files[:10]}")
if CLOCK_REPAIRS:
    loud("CLOCK DISCONTINUITIES REPAIRED (non-positive step between consecutive files):\n" + "\n".join(
        r for v in CLOCK_REPAIRS.values() for r in v["acc"] + v["temp"]))
_too_many = {b: len(v["acc"]) + len(v["temp"]) for b, v in CLOCK_REPAIRS.items()
             if len(v["acc"]) > MAX_CLOCK_REPAIRS_PER_BEARING or len(v["temp"]) > MAX_CLOCK_REPAIRS_PER_BEARING}
hook("clock_repairs_isolated", not _too_many,
     f"clock repairs per bearing: { {b: len(v['acc']) + len(v['temp']) for b, v in CLOCK_REPAIRS.items()} or 'none'} "
     f"(limit {MAX_CLOCK_REPAIRS_PER_BEARING} per modality per bearing; over the limit: {_too_many or 'none'})")
hook("monotonic_time", not _nonmono, f"per-bearing timestamps strictly increasing after midnight unwrap + clock "
     f"repair; violations: {_nonmono}; midnight wraps unwrapped: {int(INVENTORY.midnight_wraps.sum())}; isolated "
     f"clock repairs: {int(INVENTORY.clock_repairs.sum())}")

# --- span check: each modality's recorded span should match its file count x median period ---
_span_bad = []
for _b in BEARING_NAMES:
    for _key in ("acc_t", "temp_t"):
        _t = DATA[_b][_key]
        if len(_t) > 2:
            _ratio = (_t[-1] - _t[0]) / ((len(_t) - 1) * np.median(np.diff(_t)))
            if abs(_ratio - 1) > SPAN_TOL:
                _span_bad.append(f"{_b} {_key[:-2]}: span / ((n-1) x median period) = {_ratio:.3f}")
if _span_bad:
    loud("RECORDED SPAN DISAGREES WITH FILE COUNT (a gap or an unrepaired clock jump):\n" + "\n".join(_span_bad))
hook("span_vs_file_count", not _span_bad, f"every bearing's span is within {SPAN_TOL:.0%} of (n_files - 1) x median "
     f"period; exceptions: {_span_bad or 'none'}", stop=False)

# --- delimiter auto-detection ---
_semi_acc = INVENTORY[INVENTORY.acc_semicolon > 0]
_comma_temp = INVENTORY[INVENTORY.temp_comma > 0]
print("\nDelimiter auto-detection:")
print(f"  acc files: {int(INVENTORY.acc_comma.sum()):,} comma, {int(INVENTORY.acc_semicolon.sum()):,} semicolon")
for r in _semi_acc.itertuples():
    print(f"    semicolon acc files in {r.split}/{r.bearing}: {r.acc_semicolon} of {r.n_acc}")
print(f"  temp files: {int(INVENTORY.temp_semicolon.sum()):,} semicolon, {int(INVENTORY.temp_comma.sum()):,} comma")
for r in _comma_temp.itertuples():
    print(f"    comma temp files in {r.split}/{r.bearing}: {r.temp_comma} of {r.n_temp}")
_b14 = INVENTORY.set_index("bearing").loc["Bearing1_4"]
hook("delimiter_bearing1_4", _b14.acc_semicolon == _b14.n_acc > 0,
     f"Full_Test_Set/Bearing1_4: {_b14.acc_semicolon}/{_b14.n_acc} acc files detected as semicolon", stop=False)
_others = sorted(set(_semi_acc.bearing) - {"Bearing1_4"})
hook("delimiter_others", True, f"other bearings needing semicolon acc parsing: {_others or 'none'}; "
     f"temp files needing comma: {sorted(_comma_temp.bearing) or 'none'}", stop=False)

# --- acc:temp ratio (documented 10 s vs 60 s -> ~6:1) ---
_both = INVENTORY[INVENTORY.n_temp > 0]
_ratio_ok = _both["acc:temp"].between(*RATIO_TOL)
print("\nacc:temp file-count ratio (bearings with both modalities):")
for (_, r), ok in zip(_both.iterrows(), _ratio_ok):
    print(f"  {r['bearing']}: acc {r['n_acc']} / temp {r['n_temp']} = {r['acc:temp']:.2f}"
          f"{'' if ok else '   <-- outside ' + str(RATIO_TOL)}")
hook("acc_temp_ratio", _ratio_ok.all(), f"{int(_ratio_ok.sum())}/{len(_both)} bearings with both modalities have "
     f"acc:temp in {RATIO_TOL} (documented {TEMP_PERIOD_S // ACC_PERIOD_S}:1)", stop=False)

# --- zero-temperature bearings (quirk 7) ---
NO_TEMP = set(INVENTORY.loc[INVENTORY.n_temp == 0, "bearing"])
if NO_TEMP != EXPECTED_NO_TEMP:
    loud(f"ZERO-TEMPERATURE BEARING LIST DIFFERS FROM THE AUDIT.\nfound:    {sorted(NO_TEMP)}\nexpected: "
         f"{sorted(EXPECTED_NO_TEMP)}\nmissing from found: {sorted(EXPECTED_NO_TEMP - NO_TEMP)} | "
         f"unexpected: {sorted(NO_TEMP - EXPECTED_NO_TEMP)}")
hook("zero_temp_list", NO_TEMP == EXPECTED_NO_TEMP,
     f"bearings with 0 temp files: {sorted(NO_TEMP)} (expected {sorted(EXPECTED_NO_TEMP)})",
     stop=not ALLOW_NO_TEMP_MISMATCH)

# --- temp file row counts, measured (quirk 6) ---
_all_temp_len = np.concatenate([DATA[b]["temp_len"] for b in BEARING_NAMES])
TEMP_ROW_COUNTS = Counter(_all_temp_len.tolist())
if not TEMP_ROW_COUNTS:
    raise RuntimeError("No temperature files found at all.")
TEMP_LEN = int(TEMP_ROW_COUNTS.most_common(1)[0][0])
print(f"\nRows per temp file (measured, all {len(_all_temp_len):,} files): {dict(sorted(TEMP_ROW_COUNTS.items()))}")
print(f"Temperature window length = modal row count = {TEMP_LEN} samples = {TEMP_LEN / TEMP_FS:.1f} s at {TEMP_FS} Hz. "
      f"Longer files are cropped to the first {TEMP_LEN} rows; shorter files are dropped (counts in section 9).")

## Windows and degradation-stage proxy labels

A window is one record file, so it cannot cross a bearing boundary. The stage of window *i* in bearing *b* is
`min(floor(N_STAGES · (t_i − t_first(b)) / (t_last(b) − t_first(b))), N_STAGES − 1)`. It is computed **within each
bearing**, on that modality's own record timeline. For temperature, labels are computed on the bearing's full temp
timeline **before** any too-short file is dropped, so dropping a file never shifts another window's stage.

In [ ]:
# ==============================================================================
# 9. WINDOWS + PER-BEARING STAGE LABELS (+ optional vibration downsampling)
# ==============================================================================
from scipy.signal import decimate


def stage_labels(t):
    """Per-bearing equal-width elapsed-time bins. t: one bearing's record times (strictly increasing)."""
    span = t[-1] - t[0]
    assert span > 0, "a bearing needs a positive recorded span"
    frac = (t - t[0]) / span
    return np.minimum(np.floor(frac * N_STAGES).astype(np.int64), N_STAGES - 1), frac


def build_modality(modality):
    parts = defaultdict(list)
    dropped = {}
    for b in BEARING_NAMES:
        D = DATA[b]
        if modality == "vibration":
            t, nums, prefix = D["acc_t"], D["acc_num"], "acc"
            stage, frac = stage_labels(t)
            X = D["acc_x"]
            if VIB_DOWNSAMPLE_FACTOR > 1:
                X = decimate(X, VIB_DOWNSAMPLE_FACTOR, axis=1, zero_phase=True).astype(np.float32)
            keep = np.arange(len(t))
        else:
            if len(D["temp_t"]) == 0:
                continue
            t, nums, prefix = D["temp_t"], D["temp_num"], "temp"
            stage, frac = stage_labels(t)                 # full temp timeline, BEFORE dropping short files
            lens = D["temp_len"]
            offs = np.concatenate([[0], np.cumsum(lens)])
            keep = np.where(lens >= TEMP_LEN)[0]
            dropped[b] = (int((lens < TEMP_LEN).sum()), int((lens > TEMP_LEN).sum()))
            X = np.stack([D["temp_flat"][offs[i]:offs[i] + TEMP_LEN] for i in keep])[..., None]
        parts["X"].append(X)
        parts["y"].append(stage[keep])
        parts["frac"].append(frac[keep])
        parts["elapsed_s"].append((t - t[0])[keep])
        parts["file_num"].append(nums[keep])
        parts["bearing"].append(np.full(len(keep), b, dtype="<U12"))
        parts["condition"].append(np.full(len(keep), COND_OF[b], np.int64))
        parts["name"].append(np.array([f"{b}/{prefix}_{n:05d}" for n in nums[keep]]))
    M = {k: np.concatenate(v) for k, v in parts.items()}
    M["wid"] = np.arange(len(M["y"]), dtype=np.int64)
    M["bearings"] = [b for b in BEARING_NAMES if b in set(M["bearing"])]
    M["dropped"] = dropped
    return M


MOD = {m: build_modality(m) for m in ("vibration", "temperature")}
for _b in BEARING_NAMES:
    DATA[_b].pop("acc_x", None)                           # raw bursts now live in MOD["vibration"]["X"]
for _b, (_short, _long) in MOD["temperature"]["dropped"].items():
    if _short or _long:
        print(f"  temperature {_b}: {_short} files shorter than {TEMP_LEN} rows dropped, {_long} longer files cropped")

for _b, _v in CLOCK_REPAIRS.items():                    # how much the repaired clock matters for the labels
    for _kind, _key in (("acc", "acc_t"), ("temp", "temp_t")):
        if _v[_kind]:
            _st_time, _ = stage_labels(DATA[_b][_key])
            _st_order, _ = stage_labels(np.arange(len(DATA[_b][_key]), dtype=float))
            print(f"  {_b} {_kind} (clock repaired): time-based vs file-order stage labels agree on "
                  f"{(_st_time == _st_order).mean():.2%} of {len(_st_time)} files")

VIB_SEQ_LEN = MOD["vibration"]["X"].shape[1]
TEMP_SEQ_LEN = MOD["temperature"]["X"].shape[1]
N_BEARINGS = {m: len(MOD[m]["bearings"]) for m in MOD}
print(f"\nVIB_DOWNSAMPLE_FACTOR = {VIB_DOWNSAMPLE_FACTOR} -> vibration window = {VIB_SEQ_LEN} samples x "
      f"{MOD['vibration']['X'].shape[2]} channels ({ACC_ROWS / ACC_FS:.2f} s burst at {ACC_FS / VIB_DOWNSAMPLE_FACTOR:,.0f} Hz)"
      f" -> {VIB_SEQ_LEN} Mamba tokens per window")
print(f"Temperature window = {TEMP_SEQ_LEN} samples x 1 channel -> {TEMP_SEQ_LEN} Mamba tokens per window")
for m in MOD:
    print(f"{m:>11}: {len(MOD[m]['y']):,} windows from {N_BEARINGS[m]} bearings | X {MOD[m]['X'].shape} "
          f"{MOD[m]['X'].dtype} ({MOD[m]['X'].nbytes / 1e9:.2f} GB)")
hook("bearing_counts", N_BEARINGS["vibration"] == 17 and N_BEARINGS["temperature"] == 17 - len(NO_TEMP),
     f"vibration uses {N_BEARINGS['vibration']} bearings, temperature uses {N_BEARINGS['temperature']} "
     f"(vibration NOT restricted to temperature's set)")

# --- no window crosses a bearing boundary; labels are per-bearing, never pooled ---
for m, M in MOD.items():
    _blocks = [M["bearing"][0]] + [M["bearing"][i] for i in range(1, len(M["bearing"]))
                                    if M["bearing"][i] != M["bearing"][i - 1]]
    _prefix_ok = all(n.split("/")[0] == b for n, b in zip(M["name"], M["bearing"]))
    hook(f"{m}_window_bearing", _prefix_ok and len(_blocks) == len(set(_blocks)) == N_BEARINGS[m],
         f"each {m} window is one record file of one bearing; each bearing is one contiguous block")
    _ok, _table = True, []
    for b in M["bearings"]:
        sel = M["bearing"] == b
        D = DATA[b]
        t_full = D["acc_t"] if m == "vibration" else D["temp_t"]
        re_stage, _ = stage_labels(t_full)               # recomputed from THIS bearing's timestamps alone
        nums_full = D["acc_num"] if m == "vibration" else D["temp_num"]
        re_map = dict(zip(nums_full.tolist(), re_stage.tolist()))
        y_b = M["y"][sel]
        _ok &= all(re_map[n] == s for n, s in zip(M["file_num"][sel].tolist(), y_b.tolist()))
        _ok &= bool(np.all(np.diff(y_b) >= 0)) and set(y_b.tolist()) == set(range(N_STAGES))
        _ok &= re_stage[0] == 0 and re_stage[-1] == N_STAGES - 1
        _table.append([b, COND_OF[b]] + np.bincount(y_b, minlength=N_STAGES).tolist())
    hook(f"{m}_stage_labels_per_bearing", _ok,
         f"{m}: labels recomputed per bearing match, are non-decreasing in time, cover all {N_STAGES} stages, "
         f"start at Q1 and end at Q{N_STAGES} on every bearing")
    print(pd.DataFrame(_table, columns=["bearing", "cond"] + [f"Q{s + 1}" for s in range(N_STAGES)]).to_string(index=False))

In [ ]:
# ==============================================================================
# 10. LEAVE-ONE-CONDITION-OUT FOLDS + BEARING-LEVEL INNER VALIDATION
# ==============================================================================
TEMP_BEARINGS = set(MOD["temperature"]["bearings"])


def pick_val_bearings(test_c):
    """One whole bearing per TRAINING condition with >= 2 bearings. Prefer a bearing that has temperature (so both
    modalities share it) when that condition has >= 2 temperature bearings. Otherwise use a vibration-only bearing, so
    a condition's only temperature bearing stays in training. Seeded per fold; independent of the run seed."""
    rng = np.random.RandomState(INNER_VAL_SEED + test_c)
    val = []
    for tc in [c for c in FOLD_CONDITIONS if c != test_c]:
        all_b = [b for b in BEARING_NAMES if COND_OF[b] == tc]
        temp_b = [b for b in all_b if b in TEMP_BEARINGS]
        if len(temp_b) >= 2:
            pool = temp_b
        elif len(all_b) >= 2:
            pool = [b for b in all_b if b not in temp_b]
        else:
            pool = []
        if pool:
            val.append(pool[rng.randint(len(pool))])
    return sorted(val, key=BEARING_NAMES.index)


FOLDS = {}
print("Temperature bearings per condition:",
      {c: sorted(b for b in TEMP_BEARINGS if COND_OF[b] == c) for c in FOLD_CONDITIONS})
for c in FOLD_CONDITIONS:
    val_b = pick_val_bearings(c)
    for m in MOD:
        avail = MOD[m]["bearings"]
        test = [b for b in avail if COND_OF[b] == c]
        val = [b for b in avail if b in val_b]
        train = [b for b in avail if COND_OF[b] != c and b not in val_b]
        reasons = []
        if not test:
            reasons.append(f"condition {c} has no {m} bearing to test on")
        if not train:
            reasons.append(f"no {m} training bearing remains in conditions {[x for x in FOLD_CONDITIONS if x != c]}")
        if not val:
            reasons.append(f"no {m} bearing could be held out for inner validation without emptying a training condition")
        FOLDS[(m, c)] = {"train": train, "val": val, "test": test, "valid": not reasons, "reason": "; ".join(reasons)}
        assert not (set(train) & set(val) or set(train) & set(test) or set(val) & set(test))
        if not reasons:
            assert set(train) | set(val) | set(test) == set(avail)

_fold_rows = []
for (m, c), f in FOLDS.items():
    _fold_rows.append({"modality": m, "test_cond": c, "valid": f["valid"], "n_train": len(f["train"]),
                       "n_val": len(f["val"]), "n_test": len(f["test"]), "train": " ".join(f["train"]),
                       "val": " ".join(f["val"]), "test": " ".join(f["test"]), "skip_reason": f["reason"]})
with pd.option_context("display.width", 250, "display.max_colwidth", 80):
    print(pd.DataFrame(_fold_rows).to_string(index=False))
for (m, c), f in FOLDS.items():
    if not f["valid"]:
        loud(f"{m.upper()} FOLD 'test condition {c}' SKIPPED for {m} only: {f['reason']}")
hook("vibration_folds_valid", all(FOLDS[("vibration", c)]["valid"] for c in FOLD_CONDITIONS),
     "all 3 vibration folds have train / val / test bearings")
hook("temperature_folds_checked", True, f"temperature folds valid: "
     f"{[c for c in FOLD_CONDITIONS if FOLDS[('temperature', c)]['valid']]}; skipped (with reason above): "
     f"{[c for c in FOLD_CONDITIONS if not FOLDS[('temperature', c)]['valid']]}", stop=False)

## Architectures

**(A) `MambaClassifier`.** This is the prior single-modality pattern and nothing more:
- small front-end projection: Conv1d(C→64, k5) → BN → GELU → Conv1d(64→`D_MODEL`, k3) → BN → GELU, then LayerNorm;
- **2 real `mamba_ssm.Mamba` blocks** (pre-norm residual);
- attention pooling;
- a linear classifier (dropout → Linear(`D_MODEL` → `N_STAGES`)).

The front-end has no stride, so the Mamba sequence length equals the window length. The prior encoder's
modality-embedding addition is left out because it is a constant bias when there is only one modality.

**(B) `WDCNN` (vibration only).** Conv1d(2→16, **k = 64**, stride 16) → BN → ReLU → MaxPool(2), then Conv1d(16→32,
**k = 50**) → BN → ReLU, then global average pooling, then **one** Linear(32 → `N_STAGES`).

Both use class-weighted cross-entropy and AdamW with a cosine schedule, gradient clipping 1.0, 20 epochs and batch 32.
The checkpoint is the epoch with the best inner-validation macro-F1. Learning rate is 1e-4 for Mamba (the prior
recipe) and 1e-3 for the CNN (WDCNN's customary rate).

In [ ]:
# ==============================================================================
# 11. MODELS + parameter counts + real-Mamba / CNN-kernel verification
# ==============================================================================
class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model, d_state, d_conv, expand):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class MambaClassifier(nn.Module):
    """(A) small encoder -> 2 Mamba blocks -> attention pooling -> linear classifier. Input (B, L, C)."""

    def __init__(self, in_channels):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, ENC_HIDDEN, kernel_size=5, padding=2), nn.BatchNorm1d(ENC_HIDDEN), nn.GELU(),
            nn.Conv1d(ENC_HIDDEN, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = nn.ModuleList([MambaBlock(D_MODEL, MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
                                           for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, POOL_HIDDEN), nn.Tanh(), nn.Linear(POOL_HIDDEN, 1))
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(D_MODEL, N_STAGES)

    def forward(self, x):
        h = self.token_norm(self.encoder(x.transpose(1, 2)).transpose(1, 2))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier(self.dropout((h * w).sum(dim=1)))


class WDCNN(nn.Module):
    """(B) WDCNN-style compact 1D-CNN: wide first kernel, ~50-sample second kernel, global pooling, one FC layer."""

    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, CNN_C1, kernel_size=CNN_K1, stride=CNN_S1, padding=(CNN_K1 - CNN_S1) // 2)
        self.bn1 = nn.BatchNorm1d(CNN_C1)
        self.pool1 = nn.MaxPool1d(CNN_POOL1)
        self.conv2 = nn.Conv1d(CNN_C1, CNN_C2, kernel_size=CNN_K2, padding=CNN_K2 // 2)
        self.bn2 = nn.BatchNorm1d(CNN_C2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(CNN_C2, N_STAGES)

    def forward(self, x):
        h = self.pool1(F.relu(self.bn1(self.conv1(x.transpose(1, 2)))))
        h = F.relu(self.bn2(self.conv2(h)))
        return self.fc(self.gap(h).squeeze(-1))


IN_CHANNELS = {m: MOD[m]["X"].shape[2] for m in MOD}
SEQ_LEN = {"vibration": VIB_SEQ_LEN, "temperature": TEMP_SEQ_LEN}


def build_model(modality, arch):
    if arch == "mamba":
        return MambaClassifier(IN_CHANNELS[modality])
    if arch == "cnn":
        assert modality == "vibration", "the CNN baseline is vibration-only by design"
        return WDCNN(IN_CHANNELS[modality])
    raise ValueError(arch)


def n_params(model):
    return sum(p.numel() for p in model.parameters())


N_PARAMS = {}
for m, a in RUNS:
    seed_all(0)
    _model = build_model(m, a).to(device)
    N_PARAMS[(m, a)] = n_params(_model)
    with torch.no_grad():
        _model.eval()
        _out = _model(torch.randn(2, SEQ_LEN[m], IN_CHANNELS[m], device=device))
    assert _out.shape == (2, N_STAGES)
    if a == "mamba":
        _types = [type(b.mamba) for b in _model.mamba_blocks]
        hook(f"real_mamba_{m}", len(_types) == MAMBA_NUM_BLOCKS and all(t is MAMBA_CLASS for t in _types),
             f"{m} model uses {len(_types)} x {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} (no fallback)")
        print(f"  {m:>11} / mamba: {N_PARAMS[(m, a)]:>10,} params | input {SEQ_LEN[m]} x {IN_CHANNELS[m]} -> "
              f"{SEQ_LEN[m]} tokens x d_model {D_MODEL}")
    else:
        _k1 = _model.conv1.kernel_size[0]
        _l1 = (SEQ_LEN[m] + 2 * _model.conv1.padding[0] - _k1) // _model.conv1.stride[0] + 1
        hook("cnn_kernels", 64 <= _k1 <= 100 and _model.conv2.kernel_size[0] < _k1,
             f"WDCNN first-layer kernel = {_k1} (stride {_model.conv1.stride[0]}), second-layer kernel = "
             f"{_model.conv2.kernel_size[0]}")
        assert SEQ_LEN[m] >= _k1, "vibration window shorter than the CNN's first kernel (reduce VIB_DOWNSAMPLE_FACTOR)"
        print(f"  {m:>11} / cnn  : {N_PARAMS[(m, a)]:>10,} params | input {SEQ_LEN[m]} x {IN_CHANNELS[m]} -> conv1 "
              f"length {_l1} -> pool {_l1 // CNN_POOL1}")
    del _model
torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# 12. TRAINING / EVALUATION UTILITIES, results CSV (resumable)
# ==============================================================================
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score

STAGES = list(range(N_STAGES))
STAGE_NAMES = [f"Q{s + 1}" for s in STAGES]


class WindowDataset(Dataset):
    def __init__(self, M, idx):
        self.X, self.y, self.idx = M["X"], M["y"], np.asarray(idx)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return torch.from_numpy(self.X[j]), int(self.y[j]), int(j)


def make_loader(M, idx, shuffle, seed):
    return DataLoader(WindowDataset(M, idx), batch_size=BATCH_SIZE if shuffle else EVAL_BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init_fn,
                      generator=make_generator(seed) if shuffle else None)


NORM_STATS = {}


def norm_stats(modality, test_c, train_idx):
    """Per-channel mean / SD over the fold's TRAINING bearings only."""
    key = (modality, test_c)
    if key not in NORM_STATS:
        X = MOD[modality]["X"]
        s = np.zeros(X.shape[2])
        ss = np.zeros(X.shape[2])
        n = 0
        for chunk in np.array_split(train_idx, max(1, len(train_idx) // 1024)):
            x = X[chunk].astype(np.float64)
            s += x.sum(axis=(0, 1))
            ss += (x ** 2).sum(axis=(0, 1))
            n += x.shape[0] * x.shape[1]
        mu = s / n
        sd = np.sqrt(np.maximum(ss / n - mu ** 2, 0.0)) + NORM_EPS
        NORM_STATS[key] = (mu.astype(np.float32), sd.astype(np.float32))
    return NORM_STATS[key]


def compute_metrics(y, P):
    pred = P.argmax(1)
    cm = confusion_matrix(y, pred, labels=STAGES)
    rec = np.where(cm.sum(1) > 0, np.diag(cm) / np.maximum(cm.sum(1), 1), np.nan)
    out = {"accuracy": float(accuracy_score(y, pred)),
           "macro_f1": float(f1_score(y, pred, average="macro", labels=STAGES, zero_division=0)),
           "balanced_accuracy": float(balanced_accuracy_score(y, pred))}
    out.update({f"recall_{STAGE_NAMES[s]}": float(rec[s]) for s in STAGES})
    out["confusion_matrix"] = cm
    return out


@torch.no_grad()
def predict(model, loader, mu, sd):
    model.eval()
    probs, ys, wids = [], [], []
    for x, y, j in loader:
        x = (x.to(device, non_blocking=True) - mu) / sd
        probs.append(F.softmax(model(x), dim=-1).float().cpu().numpy())
        ys.append(y.numpy())
        wids.append(j.numpy())
    return np.concatenate(probs), np.concatenate(ys), np.concatenate(wids)


def run_tag(modality, arch, test_c, seed):
    return f"{modality}_{arch}_testC{test_c}_{seed}"


def probs_path(modality, arch, test_c, seed):
    return PROBS_DIR / f"{run_tag(modality, arch, test_c, seed)}.npz"


def ckpt_path(modality, arch, test_c, seed):
    return CKPT_DIR / f"{run_tag(modality, arch, test_c, seed)}.pt"


def done_keys():
    """Runs whose CSV row AND prediction file both exist under the current CONFIG_TAG."""
    if not RESULTS_CSV.exists():
        return set()
    df = pd.read_csv(RESULTS_CSV)
    keys = set()
    for r in df[df.config_tag == CONFIG_TAG].itertuples():
        p = probs_path(r.modality, r.arch, int(r.test_condition), int(r.seed))
        if p.exists():
            with np.load(p, allow_pickle=False) as z:
                if str(z["config_tag"]) == CONFIG_TAG:
                    keys.add((r.modality, r.arch, int(r.test_condition), int(r.seed)))
    return keys


def append_result(row):
    new = pd.DataFrame([row])
    if RESULTS_CSV.exists():
        old = pd.read_csv(RESULTS_CSV)
        stale = ((old.config_tag == row["config_tag"]) & (old.modality == row["modality"]) & (old.arch == row["arch"])
                 & (old.test_condition == row["test_condition"]) & (old.seed == row["seed"]))
        new = pd.concat([old[~stale], new], ignore_index=True)
    tmp = RESULTS_CSV.with_suffix(".tmp")
    new.to_csv(tmp, index=False)
    tmp.replace(RESULTS_CSV)


def run_one(modality, arch, test_c, seed):
    t0 = time.time()
    M, fold = MOD[modality], FOLDS[(modality, test_c)]
    idx = {s: np.where(np.isin(M["bearing"], fold[s]))[0] for s in ("train", "val", "test")}
    mu_np, sd_np = norm_stats(modality, test_c, idx["train"])
    mu = torch.tensor(mu_np, device=device).view(1, 1, -1)
    sd = torch.tensor(sd_np, device=device).view(1, 1, -1)

    seed_all(seed)                                           # at the START of every run, before the model is built
    model = build_model(modality, arch).to(device)
    if arch == "mamba":
        assert all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks)
    counts = np.bincount(M["y"][idx["train"]], minlength=N_STAGES)
    assert (counts > 0).all(), f"a stage is missing from training: {counts}"
    cw = torch.tensor(len(idx["train"]) / (N_STAGES * counts), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE[arch], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(M, idx["train"], True, seed)
    va = make_loader(M, idx["val"], False, seed)
    te = make_loader(M, idx["test"], False, seed)
    tag = f"[{modality}/{arch} test=C{test_c} seed {seed}]"
    print(f"{tag} {N_PARAMS[(modality, arch)]:,} params | bearings train {len(fold['train'])} / val "
          f"{len(fold['val'])} {fold['val']} / test {len(fold['test'])} | windows train {len(idx['train'])} / val "
          f"{len(idx['val'])} / test {len(idx['test'])} | seq len {SEQ_LEN[modality]}")

    best_f1, best_state, best_epoch = -1.0, None, None
    for epoch in range(1, EPOCHS + 1):
        te0 = time.time()
        model.train()
        tot, nb = 0.0, 0
        for x, y, _ in tr:
            x = (x.to(device, non_blocking=True) - mu) / sd
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            tot += loss.item()
            nb += 1
        scheduler.step()
        if not math.isfinite(tot):
            raise RuntimeError(f"{tag} non-finite training loss at epoch {epoch}")
        vP, vy, _ = predict(model, va, mu, sd)
        vf1 = compute_metrics(vy, vP)["macro_f1"]
        if vf1 > best_f1:
            best_f1, best_state, best_epoch = vf1, copy.deepcopy(model.state_dict()), epoch
        print(f"  epoch {epoch:2d} | loss {tot / nb:.4f} | val macro-F1 {vf1:.4f}{' *' if epoch == best_epoch else ''} "
              f"| {time.time() - te0:.0f} s")
        if epoch == 1:
            print(f"  (estimated run time ≈ {(time.time() - te0) * EPOCHS / 60:.1f} min)")
        if epoch - best_epoch >= EARLY_STOP_PATIENCE:
            print(f"  early stop at epoch {epoch}, no improvement for {EARLY_STOP_PATIENCE} epochs")
            break

    model.load_state_dict(best_state)
    P, y, wid = predict(model, te, mu, sd)
    assert np.array_equal(wid, idx["test"]) and np.array_equal(y, M["y"][wid])
    met = compute_metrics(y, P)
    np.savez(probs_path(modality, arch, test_c, seed),
             window_id=wid, window_name=M["name"][wid], bearing=M["bearing"][wid], condition=M["condition"][wid],
             stage_label=y, probs=P.astype(np.float32), pred=P.argmax(1), frac_of_life=M["frac"][wid],
             file_num=M["file_num"][wid], config_tag=np.array(CONFIG_TAG), modality=np.array(modality),
             arch=np.array(arch), test_condition=np.int64(test_c), seed=np.int64(seed))
    torch.save({"state_dict": best_state, "best_epoch": best_epoch, "best_val_macro_f1": best_f1,
                "norm_mu": mu_np, "norm_sd": sd_np, "fold": fold, "config_tag": CONFIG_TAG,
                "modality": modality, "arch": arch, "test_condition": test_c, "seed": seed},
               ckpt_path(modality, arch, test_c, seed))
    row = {"config_tag": CONFIG_TAG, "modality": modality, "arch": arch, "test_condition": test_c, "seed": seed,
           "n_bearings_modality": N_BEARINGS[modality], "n_train_bearings": len(fold["train"]),
           "n_val_bearings": len(fold["val"]), "n_test_bearings": len(fold["test"]),
           "train_bearings": " ".join(fold["train"]), "val_bearings": " ".join(fold["val"]),
           "test_bearings": " ".join(fold["test"]), "n_train_windows": len(idx["train"]),
           "n_val_windows": len(idx["val"]), "n_test_windows": len(idx["test"]), "seq_len": SEQ_LEN[modality],
           "n_params": N_PARAMS[(modality, arch)], "best_epoch": best_epoch, "best_val_macro_f1": best_f1,
           **{k: v for k, v in met.items() if k != "confusion_matrix"},
           "confusion_matrix": json.dumps(met["confusion_matrix"].tolist()),
           "runtime_s": round(time.time() - t0, 1), "finished_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    append_result(row)
    print(f"{tag} TEST: acc {met['accuracy']:.4f} | macro-F1 {met['macro_f1']:.4f} | bal-acc "
          f"{met['balanced_accuracy']:.4f} | recall " + " ".join(f"{n} {met['recall_' + n]:.3f}" for n in STAGE_NAMES)
          + f" | best epoch {best_epoch} | {time.time() - t0:.0f} s")
    del model, best_state
    torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# 13. RUN ALL (resumable: runs already logged under this CONFIG_TAG are skipped)
# ==============================================================================
# Only each hook's LATEST result counts: a failure fixed by re-running its cell no longer blocks training.
_hard_fail = [n for n, v in HOOKS.items() if (not v[-1][0]) and v[-1][2]]
if _hard_fail:
    raise RuntimeError(f"Hard verification hooks failed: {_hard_fail} — not training.")

TRAIN_T0 = time.time()
_done = done_keys()
_n_total = sum(len(SEEDS) for c in FOLD_CONDITIONS for m, a in RUNS if FOLDS[(m, c)]["valid"])
print(f"{_n_total} runs in the matrix; {len(_done)} already complete under {CONFIG_TAG}")
for c in FOLD_CONDITIONS:
    for m, a in RUNS:
        if not FOLDS[(m, c)]["valid"]:
            print(f"SKIP {m}/{a} test=C{c}: {FOLDS[(m, c)]['reason']}")
            continue
        for s in SEEDS:
            if (m, a, c, s) in _done:
                print(f"skip (already logged) {run_tag(m, a, c, s)}")
                continue
            run_one(m, a, c, s)
print(f"Training section finished in {(time.time() - TRAIN_T0) / 60:.1f} min (this session)")

## Results

**Per condition**: the fold that tests on that condition, as mean ± SD over seeds. **Pooled**: for each seed, the
test predictions of all valid folds are concatenated (every bearing of that modality is tested exactly once) and the
metrics are computed on the pooled windows. Mean ± SD over seeds is reported. Every number carries the bearing count
it is computed on. Chance level for balanced accuracy is 1 / `N_STAGES`.

In [ ]:
# ==============================================================================
# 14. RESULTS TABLES — per condition and pooled; vibration Mamba vs CNN; temperature Mamba
# ==============================================================================
RES = pd.read_csv(RESULTS_CSV)
RES = RES[RES.config_tag == CONFIG_TAG].copy()
METRICS = ["accuracy", "macro_f1", "balanced_accuracy"] + [f"recall_{n}" for n in STAGE_NAMES]


def pm(x):
    x = np.asarray(x, float)
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else (f"{x[0]:.4f}" if len(x) else "-")


print(f"Chance balanced accuracy = {1 / N_STAGES:.3f}.  Bearings: vibration {N_BEARINGS['vibration']}, "
      f"temperature {N_BEARINGS['temperature']}.  Seeds {SEEDS}.\n")

# ---------------- per condition ----------------
_pc_rows = []
for (m, a, c), g in RES.groupby(["modality", "arch", "test_condition"], sort=True):
    _pc_rows.append({"modality": m, "arch": a, "level": f"test C{c} ({CONDITIONS[c]})",
                     "bearings": f"{int(g.n_test_bearings.iloc[0])} of {N_BEARINGS[m]} ({g.test_bearings.iloc[0]})",
                     "seeds": len(g), **{k: pm(g[k]) for k in METRICS}})
for (m, c), f in FOLDS.items():
    if not f["valid"]:
        _pc_rows.append({"modality": m, "arch": "mamba", "level": f"test C{c}", "bearings": "SKIPPED",
                         "seeds": 0, **{k: f["reason"] if k == "accuracy" else "" for k in METRICS}})

# ---------------- pooled over folds ----------------
_pool_rows, POOLED_CM = [], {}
for m, a in RUNS:
    valid_c = [c for c in FOLD_CONDITIONS if FOLDS[(m, c)]["valid"]]
    per_seed, cms = [], []
    for s in SEEDS:
        paths = [probs_path(m, a, c, s) for c in valid_c]
        if not all(p.exists() for p in paths):
            print(f"  pooled {m}/{a} seed {s}: incomplete ({sum(p.exists() for p in paths)}/{len(paths)} folds) — omitted")
            continue
        ys, Ps, bs = [], [], []
        for p in paths:
            with np.load(p, allow_pickle=False) as z:
                ys.append(z["stage_label"]); Ps.append(z["probs"]); bs.append(z["bearing"])
        met = compute_metrics(np.concatenate(ys), np.concatenate(Ps))
        met["n_bearings"] = len(set(np.concatenate(bs).tolist()))
        per_seed.append(met)
        cms.append(met["confusion_matrix"])
    if not per_seed:
        continue
    POOLED_CM[(m, a)] = np.mean(cms, axis=0)
    nb = per_seed[0]["n_bearings"]
    _pool_rows.append({"modality": m, "arch": a, "level": f"POOLED ({len(valid_c)} folds)",
                       "bearings": f"{nb} of {N_BEARINGS[m]}", "seeds": len(per_seed),
                       **{k: pm([d[k] for d in per_seed]) for k in METRICS}})

TABLE = pd.DataFrame(_pool_rows + _pc_rows)
with pd.option_context("display.width", 300, "display.max_columns", 30, "display.max_colwidth", 70):
    for m in ("vibration", "temperature"):
        print(f"===== {m.upper()} ({N_BEARINGS[m]} bearings) =====")
        print(TABLE[TABLE.modality == m].to_string(index=False))
        print()
TABLE.to_csv(SUMMARY_CSV, index=False)

# ---------------- pooled confusion matrices (mean counts over seeds; rows = true stage) ----------------
for (m, a), cm in POOLED_CM.items():
    print(f"Pooled confusion matrix — {m}/{a} ({N_BEARINGS[m]} bearings), mean over seeds; rows true, cols predicted")
    print(pd.DataFrame(np.round(cm, 1), index=[f"true {n}" for n in STAGE_NAMES],
                       columns=[f"pred {n}" for n in STAGE_NAMES]).to_string())
    print(pd.DataFrame(np.round(cm / cm.sum(1, keepdims=True), 3), index=[f"true {n}" for n in STAGE_NAMES],
                       columns=[f"pred {n}" for n in STAGE_NAMES]).to_string(), "\n")

# ---------------- vibration: Mamba vs CNN (paired by fold x seed; descriptive only) ----------------
_v = RES[RES.modality == "vibration"].pivot_table(index=["test_condition", "seed"], columns="arch", values="macro_f1")
if {"mamba", "cnn"} <= set(_v.columns):
    _v["mamba_minus_cnn"] = _v["mamba"] - _v["cnn"]
    print("Vibration macro-F1, Mamba vs WDCNN, paired by (test condition, seed) — 17 bearings:")
    print(_v.round(4).to_string())
    print(f"mean Mamba − CNN = {_v.mamba_minus_cnn.mean():+.4f} over {len(_v)} pairs "
          f"(descriptive; {len(SEEDS)} seeds x 3 folds is a search-phase sample, not a significance test)")
print(f"\nParameter counts: " + " | ".join(f"{m}/{a} {N_PARAMS[(m, a)]:,}" for m, a in RUNS))
print(f"Saved: {RESULTS_CSV} (per run), {SUMMARY_CSV} (tables), {PROBS_DIR}/ (per-window test probabilities), "
      f"{CKPT_DIR}/ (best checkpoints)")

In [ ]:
# ==============================================================================
# 15. DELIVERABLES SUMMARY — hooks, data checks, runtime
# ==============================================================================
print("VERIFICATION HOOKS")
for name, entries in HOOKS.items():
    for ok, msg, stop in entries[-1:]:              # latest result of each hook
        print(f"  {'PASS' if ok else 'FAIL'}{'' if stop else ' (soft)'} | {name}: {msg}")

print("\nDELIVERABLES")
print(f"  Recursive extraction: {len(EXTRACTION_LOG)} archives over {_n_passes} passes, 0 .zip left; "
      f"17 bearings from {USE_SPLITS}; {EXCLUDED_SPLIT}/ excluded, files read from it: 0 "
      f"(files read by split: {dict(READ_COUNT)})")
print(f"  Delimiter auto-detection: Bearing1_4 acc semicolon {int(_b14.acc_semicolon)}/{int(_b14.n_acc)}; other "
      f"semicolon-acc bearings: {_others or 'none'}; comma temp files: {sorted(_comma_temp.bearing) or 'none'}")
print(f"  Zero-temperature bearings: {sorted(NO_TEMP)} -> {'MATCHES' if NO_TEMP == EXPECTED_NO_TEMP else 'DIFFERS FROM'}"
      f" the audit list")
print(f"  Clock discontinuities repaired: {CLOCK_REPAIRS and {b: {k: len(x) for k, x in v.items()} for b, v in CLOCK_REPAIRS.items()} or 'none'}")
print(f"  Temp rows per file (measured): {dict(sorted(TEMP_ROW_COUNTS.items()))}; window length {TEMP_LEN}")
print(f"  Bearings: vibration {N_BEARINGS['vibration']}, temperature {N_BEARINGS['temperature']}; temperature folds "
      f"skipped: {[c for c in FOLD_CONDITIONS if not FOLDS[('temperature', c)]['valid']] or 'none'}")
print(f"  Parameters: " + " | ".join(f"{m}/{a} {N_PARAMS[(m, a)]:,}" for m, a in RUNS)
      + f" | WDCNN first kernel {CNN_K1}, second {CNN_K2}")
print(f"  Sequence lengths: vibration {VIB_SEQ_LEN} (VIB_DOWNSAMPLE_FACTOR {VIB_DOWNSAMPLE_FACTOR}), temperature {TEMP_SEQ_LEN}")
_logged = RES.runtime_s.sum() / 60 if len(RES) else 0.0
print(f"  Runs logged under {CONFIG_TAG}: {len(RES)} | summed training time of logged runs {_logged:.1f} min")
print(f"  Total notebook wall time this session: {(time.time() - NOTEBOOK_T0) / 60:.1f} min")
print("  External dependencies: none beyond pip (mamba-ssm, causal-conv1d) and the PHM Society download URL.")